# 02 - Phase 2: ViT Integration, the Three Bugs, and the Fixes

**One-liner:** swap the ResNet-101 backbone for a plain ViT-B/16, keep the rest of Faster R-CNN + STTran intact. This is the phase that exposed three serious bugs we had to fix before *any* number was trustworthy.

This notebook is the heart of your presentation story. Spend the most time here.

## The Phase 2 architecture

```mermaid
flowchart LR
    Frame[AG frame, 1024x1024 LSJ] --> ViT["plain ViT-B/16 from timm"]
    ViT --> Bridge[ViTDet synthetic FPN]
    Bridge --> P["p2 / p3 / p4 / p5"]
    P --> RPN[RPN]
    RPN --> RoI[RoI Align]
    RoI --> Heads["RCNN_cls + RCNN_bbox (reinitialized)"]
    Heads --> STTran[STTran heads]
    STTran --> Out["per-frame scene graph"]
```

Compared to Phase 1, exactly two things change:
1. ResNet-101 -> ViT-B/16
2. The neck becomes a synthetic ViTDet pyramid because plain ViT outputs only one scale

Everything else - RPN, RoI Align, RCNN cls/bbox heads, STTran head - is the same code as Phase 1.

**Repo files for this phase:**
- `phase2_vitdet/simple_vitdet_fpn.py` - the synthetic pyramid bridge
- `phase2_vitdet/adapter.py` - thin wrapper exposing it to `object_detector.py`
- `STTran/lib/object_detector.py` - the `vitdet` branch of `_extract_base_features`
- `STTran/train_detector_stage1.py` - standalone Stage 1 detector trainer
- `STTran/train.py` - Stage 2 graph trainer (BF16 + Poison-Pill detection lives here)

## The ViTDet bridge in code

Plain ViT outputs a single feature map at stride 1/16. Faster R-CNN expects four levels (`p2`/`p3`/`p4`/`p5` at strides 1/4, 1/8, 1/16, 1/32). The bridge synthesizes the missing levels with simple up/downsample convolutions.

From `phase2_vitdet/simple_vitdet_fpn.py` lines 81-98:

```python
self.p2 = nn.Sequential(
    nn.ConvTranspose2d(self.embed_dim, out_channels, kernel_size=2, stride=2),  # upsample
    nn.GELU(),
    nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
    nn.GELU(),
)
self.p3 = nn.Sequential(
    nn.Conv2d(self.embed_dim, out_channels, kernel_size=1),                      # identity scale
    nn.GELU(),
)
self.p4 = nn.Sequential(
    nn.Conv2d(self.embed_dim, out_channels, kernel_size=3, stride=2, padding=1), # downsample
    nn.GELU(),
)
self.p5 = nn.Sequential(
    nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=2, padding=1),   # further downsample
    nn.GELU(),
)
```

And in `forward()` (lines 207-219):

```python
base = self._reshape_tokens(tokens, h, w)  # [B, C, H/16, W/16]
p3 = self.p3(base)
p2 = self.p2(base)
p4 = self.p4(base)
p5 = self.p5(p4)
raw = {"p2": p2, "p3": p3, "p4": p4, "p5": p5}
```

**Mathematically**: $F_s = U_s(f_\theta(I))$ for $s \in \{1/4, 1/8, 1/16, 1/32\}$ where $f_\theta$ is the ViT and $U_s$ is the per-level convolutional resampler.

**The 1x1 align layer** at line 73-78 is what reshapes `p3` from `out_channels=256` to `align_channels=1024` so the inherited `RCNN_cls_score` can consume it (the ResNet-101 RCNN head expects 1024-channel pooled features). This is the key bit of plumbing that lets us reuse the entire Faster R-CNN shell unchanged.

## Why ViT instead of ResNet?

From the proposal (`knowledge information/st_project_proposal.pdf`):

> Our hypothesis states that ViTs' global self-attention mechanism will create superior visual features, which will enhance relationship detection capabilities.

Concrete reasons:
- **Long-range dependencies**: ResNet only sees a few pixels in early layers. A ViT's first layer already attends globally. Spatio-temporal relations like *person on the left, cup on the right, holding* benefit.
- **Single-scale, simple**: a ViT is a clean stack of identical blocks. No anchor-pyramid stage hand-engineering. ViTDet showed the synthetic-pyramid trick is enough.
- **Foundation-model on-ramp**: once you have a ViT-shaped backbone slot, you can drop in DINOv2 / MAE / etc. without rewiring anything. This matters for Phase 3.

**Reality check:** the *trained-from-scratch* ViT did NOT beat ResNet on AG. Phase 2's role is to provide the corrected scaffolding; the win comes in Phase 3 when we drop in DINOv2's pretrained features.

## The first set of Phase 2 results was broken (and not in obvious ways)

Early Phase 2 runs produced *suspicious* numbers. PredCLS looked great, SGCLS was 0.013 R@20 (essentially zero), SGDET was middling. We knew something was off but the failure modes hid each other.

Three independent bugs were stacked. We had to peel them off one by one. From `knowledge information/sttran_backbone_migration_report_latest.pdf`:

> The result was a classic false positive success pattern: the old checkpoint could look very strong on PredCLS when ground-truth object labels were given, while SGCLS and SGDET collapsed because the actual object/detector path was broken.

Below: each bug, the symptom, the fix, and the code location.

## Bug 1: BF16 "Poison Pill" - loss collapses to NaN/Inf

### Symptom
Training loss randomly explodes to `NaN` or `Inf` mid-epoch. The whole batch is wasted. After enough poisoned batches, the optimizer state itself is corrupted and the run is dead.

### Why it happens
On H100s, we run mixed precision with **BF16 autocast**. BF16 has 8-bit exponent (great range) but only **7-bit mantissa** (terrible precision).

Inside cross-entropy loss we evaluate $-\log \hat{p}$. When $\hat{p}$ is small (object class with low predicted probability), $-\log \hat{p}$ blows up - and small precision errors in $\hat{p}$ become huge errors in $-\log \hat{p}$. Same for $\log \sigma(z_c)$ inside relation BCE.

Result: BF16 logits go through softmax/sigmoid in BF16, the log term saturates or overflows, and the loss returns `NaN`.

### The fix
Cast logits to FP32 *before* the non-linearity and the loss. Formally:

$$\hat y = \mathrm{softmax}(\mathrm{cast}_{\text{FP32}}(z))$$
$$\mathcal{L}_{\text{obj}} = -\sum_c y_c \log \hat y_c$$

**In code, `STTran/train.py` lines 994-1003:**

```python
# Compute relation losses in FP32 for numerical stability even when
# model forward runs under AMP/BF16.
attention_distribution = pred['attention_distribution'].float()
spatial_distribution = pred['spatial_distribution'].float()
contact_distribution = pred['contacting_distribution'].float()

attention_label = torch.tensor(
    pred['attention_gt'],
    dtype=torch.long,
    device=attention_distribution.device,
).squeeze()
```

Three lines, but it is *the* fix. The backbone forward stays BF16 (fast). Loss stays FP32 (stable). H100 is happy.

**Defense-in-depth**: even with the cast, the launcher still detects any `NaN`/`Inf` loss and skips the batch (`STTran/train.py` lines 1055-1078):

```python
local_non_finite = not bool(torch.isfinite(loss).item())
if not local_non_finite:
    for loss_value in losses.values():
        if not bool(torch.isfinite(loss_value).item()):
            local_non_finite = True
            break
...
if poison_batch:
    print('\n[WARNING] Poison Pill Caught! NaN/Inf detected at epoch {}, batch {}. ...')
```

**Why we call it the "Poison Pill"**: even one `NaN` in the loss poisons the *entire* batch's gradient (because `NaN` propagates through every parameter update). Catching it at the batch level lets us skip and continue.

### Additional sub-fix: Stage 1 was contaminated

Original Stage 1 still computed relation losses and skipped batches when those went `NaN`, even when the *object* loss was fine. This silently dropped detector gradients for affected frames. From the migration report:

> Stage 1 was rewritten to compute object loss only.

After this: object semantics stopped collapsing, ROI top-1 hit climbed back into the 80-86% range, and the "poison pill" batch-skip count fell to zero in Stage 1.

## Bug 2: BatchNorm vs GroupNorm - 209 detector keys silently dropped at eval

### Symptom
Detector mAP goes to **zero** at evaluation time, even though training looked fine and the checkpoint was saved without errors. Every box returns garbage.

### Why it happens
Faster R-CNN ships with BatchNorm layers. BN buffers (`running_mean`, `running_var`) are saved into the state dict. ViT-based detectors typically use GroupNorm (no buffers, deterministic at eval, doesn't need a large effective batch).

If we trained with `DETECTOR_BN_MODE=groupnorm` but evaluated with the env var unset (defaulting to `batchnorm`), the eval-time module would have a different module tree from the training-time module, and `load_state_dict(strict=False)` would silently drop **209 keys** - every BN running-buffer pair across the backbone-neck-RPN stack. That left the eval-time BN layers with random-init buffers, which gives garbage output.

### The fix
Force `DETECTOR_BN_MODE=groupnorm` consistently across train and eval, and write a robust `_replace_batchnorm_with_groupnorm` helper.

**Helper**: `STTran/lib/object_detector.py` lines 32-50:

```python
def _replace_batchnorm_with_groupnorm(module, preferred_groups=32):
    replaced = 0
    for child_name, child in list(module.named_children()):
        replaced += _replace_batchnorm_with_groupnorm(child, preferred_groups=preferred_groups)
        if isinstance(child, nn.BatchNorm2d):
            num_groups = _valid_group_count(child.num_features, preferred_groups)
            gn = nn.GroupNorm(
                num_groups=num_groups,
                num_channels=child.num_features,
                eps=child.eps,
                affine=child.affine,
            )
            if child.affine:
                with torch.no_grad():
                    gn.weight.copy_(child.weight)
                    gn.bias.copy_(child.bias)
            setattr(module, child_name, gn)
            replaced += 1
    return replaced
```

**Application** (lines 187-206):

```python
detector_bn_mode = os.environ.get('DETECTOR_BN_MODE', 'batchnorm').strip().lower()
detector_gn_groups = int(os.environ.get('DETECTOR_GN_GROUPS', '32'))
if detector_bn_mode == 'groupnorm':
    replaced_bn = _replace_batchnorm_with_groupnorm(
        self.fasterRCNN, preferred_groups=detector_gn_groups
    )
    print(
        'detector norm mode: groupnorm (groups={}); replaced BatchNorm2d layers: {}'.format(
            detector_gn_groups, replaced_bn
        )
    )
```

**And the launchers now default to groupnorm for ViT/DINOv2** (`STTran/scripts/eval_sttran_h100.sbatch:43-51`, `train_phase3_dinov2_h100.sbatch:91`).

### Mathematical sketch
GroupNorm with $g=32$ groups normalizes each group of channels within one sample:
$$\mathrm{GN}_{32}(x) = \gamma \odot \frac{x - \mu_g}{\sqrt{\sigma_g^2 + \varepsilon}} + \beta$$
where $\mu_g$ and $\sigma_g^2$ are the per-group statistics. **No running buffers** = no train/eval drift = no 209-key loss.

## Bug 3: The Mixed-Path Bug - PredCLS/SGCLS used legacy ResNet, SGDET used ViT

### Symptom
Phase 2 Epoch 0 ViT model showed:
- PredCLS R@20 = ~44 (looks reasonable)
- SGCLS R@20 = ~0.013 (essentially zero)
- SGDET R@20 = ~16 (low but plausible)

Looked like the model could classify *predicates* but couldn't classify *objects*. Confusing because PredCLS uses the same object features as SGCLS.

### Why it happens
Buried inside `STTran/lib/object_detector.py`, the GT-box branch (used by PredCLS and SGCLS, since they get ground-truth boxes) was extracting features through the *legacy ResNet base*, even when `backbone_name = 'vitdet'`. The SGDET branch (which detects boxes) had been correctly rewired to call the new ViT path. The two branches had diverged.

The result: PredCLS/SGCLS were silently scoring against a ResNet-101 trained on AG (which is fine - but it has nothing to do with our ViT experiment), while SGDET scored against the new ViT path. **They were not even the same model.** Comparing them was meaningless.

### The fix
Force every mode through `self._extract_base_features(...)`, which dispatches on `backbone_name` to either the ViT path (`self.vitdet(images)['base']`) or the legacy ResNet path (`self.fasterRCNN.RCNN_base(images)`).

**`STTran/lib/object_detector.py` lines 758-771 (the GT-box branch's feature extraction):**

```python
counter = 0
FINAL_BASE_FEATURES = torch.empty(0, device=device)

while counter < im_data.shape[0]:
    if counter + self.detector_chunk < im_data.shape[0]:
        inputs_data = im_data[counter:counter + self.detector_chunk]
    else:
        inputs_data = im_data[counter:]
    # Keep PredCLS/SGCLS on the same backbone path as SGDET so ViT
    # checkpoints do not silently fall back to the legacy ResNet base.
    base_feat = self._extract_base_features(inputs_data)
    FINAL_BASE_FEATURES = torch.cat((FINAL_BASE_FEATURES, base_feat), 0)
    counter += self.detector_chunk
```

Notice the comment - it documents the bug and its fix.

### The mode-consistency invariant
After the fix, the invariant we now enforce is:

$$\forall m \in \{\text{PredCLS}, \text{SGCLS}, \text{SGDET}\},\ f_\theta^{(m)} \equiv f_\theta$$

i.e. all three modes use the *same* backbone. In Phase 3 we additionally enforce a runtime assertion (`PHASE3_ASSERT_DINOV2_PATH=1`) that the DINOv2 path is actually being used.

### Impact
Corrected SGCLS R@20 jumped from **0.013 -> 0.297** on the same checkpoint, immediately proving the old SGCLS numbers were obsolete. From the migration report:

> All PredCLS and SGCLS numbers collected before the backbone-path fix should be treated as obsolete for ViT comparison.

## Companion fix: SGCLS-specific destructive duplicate-suppression

Even after the Mixed-Path fix, SGCLS had its own residual bug: the legacy SGCLS branch had a duplicate-relabel heuristic that could *change valid non-human boxes into person predictions* when two boxes overlapped. This deleted correct ground-truth-aligned object assignments.

**Code location:** `STTran/lib/sttran.py` lines 49-69 introduce the configurable policy:

```python
self.sgcls_duplicate_policy = os.environ.get('SGCLS_DUPLICATE_POLICY', 'legacy').strip().lower()
policy_aliases = {
    'current': 'legacy',
    'legacy_mode': 'legacy',
    'off': 'none',
    'disabled': 'none',
    'iou_only': 'iou',
    'soft_iou': 'iou',
}
self.sgcls_duplicate_policy = policy_aliases.get(
    self.sgcls_duplicate_policy,
    self.sgcls_duplicate_policy,
)
...
self.sgcls_label_source = os.environ.get('SGCLS_LABEL_SOURCE', 'decoder').strip().lower()
```

**Defaults locked in launchers** (`STTran/scripts/eval_sttran_h100.sbatch:121-122`):

```bash
export SGCLS_DUPLICATE_POLICY="${SGCLS_DUPLICATE_POLICY:-iou}"
export SGCLS_LABEL_SOURCE="${SGCLS_LABEL_SOURCE:-detector}"
```

**Translation**:
- `SGCLS_DUPLICATE_POLICY=iou`: deduplicate using IoU only, never relabel an object as a person.
- `SGCLS_LABEL_SOURCE=detector`: when reading the per-box object class, use the upstream Faster R-CNN classifier head rather than the in-`ObjectClassifier` mini-decoder, which had been misbehaving on AG.

After this change, SGCLS finally became a trustworthy read of the corrected backbone path.

## Order of attack and how we knew the fixes were real

We fixed them in this order:
1. **Bug 2 first** (GroupNorm/BatchNorm consistency) - fast to test, ruled out the eval-load explanation. Symptom: detector mAP went from 0 back to ~22.
2. **Bug 1 second** (BF16 Poison Pill + Stage-1 contamination) - made training stable enough to actually produce checkpoints. Symptom: poison-pill batch-skips fell to zero, ROI top-1 climbed to 86%.
3. **Bug 3 third** (Mixed-Path) - the moment we suspected PredCLS and SGCLS were silently using the wrong backbone, we checked the GT-box branch and saw it. Symptom: SGCLS R@20 jumped from 0.013 -> 0.297 immediately, on the *same* checkpoint.

**Reality check:** the Phase 2 corrected baseline (mAP 23.19, SGCLS R@20 30.75) is *not* better than Phase 1 - it's actually slightly behind. That's fine. Phase 2's job was to (a) prove the harness works for any backbone, (b) be the corrected scaffold that Phase 3 inherits. The ablation only becomes interesting in Phase 3.

## Final Phase 2 corrected scorecard

From the final report and `Big Picture.pdf`:

| Metric | Value |
|---|---|
| Detector mAP@0.5 | 23.19 |
| PredCLS R@20 (With) | 44.95 |
| SGCLS R@20 (With) | 30.75 |
| SGDET R@20 (With) | 23.07 |

These are the first *honest* ViT numbers. They trail the base paper, which is consistent with training a ViT *from scratch* on a small AG-only detector corpus - not enough data to get full ViT advantages.

**This is exactly the motivation for Phase 3.** If we don't have enough data to train a ViT well, use a *frozen* foundation-model ViT that has already been trained on 142M images.

## How to talk about Phase 2 in 60 seconds

> "Phase 2 swapped the ResNet for a plain ViT-B/16 with a ViTDet synthetic feature pyramid. Three bugs surfaced. First, BF16 mixed precision was silently underflowing inside the loss - we cast the logits to FP32 before softmax/sigmoid and the NaN poison-pill problem went away. Second, BatchNorm and GroupNorm settings drifted between train and eval, dropping 209 detector keys at load time and zeroing out mAP - we standardized on GroupNorm everywhere. Third, and most important, PredCLS and SGCLS were quietly using the legacy ResNet feature path while SGDET used the new ViT path - the cross-mode comparison was meaningless. We rewired all three modes through one `_extract_base_features` call. SGCLS R@20 jumped from 0.013 to 0.297 on the same checkpoint. After the fixes, Phase 2 was an honest ViT baseline at 23.19 mAP, but the trained-from-scratch ViT didn't beat ResNet on its own. That's exactly what motivated Phase 3 - drop in a frozen foundation backbone instead."

Practice that paragraph until it flows. It's the answer to half the possible Q&A.

---
**Next:** open `03_phase3_dinov2_foundation.ipynb` for the win.